# Neural-Flow: Continuous-Time Brain State & Connectome Dynamics
### Graph Neural ODEs and Optimal Transport Conditional Flow Matching (OT-CFM)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/neural-flow/neural-flow-connectome/blob/main/notebooks/Neural_Flow_Interactive_Demo.ipynb)

This interactive notebook demonstrates the complete end-to-end continuous-time neural dynamics framework:
1. **Structural Connectome Graph**: $N=68$ Desikan-Killiany parcellation, symmetric normalized Laplacian $\mathcal{L}_{\mathrm{sym}}$, and spectral bounds $\lambda_i \in [0, 2]$.
2. **Continuous-Time Graph Neural ODE**: Solving $\frac{dX}{dt} = -\alpha X - \kappa \mathcal{L}_{\mathrm{sym}} X + f_\theta(X, \mathcal{L}_{\mathrm{sym}}, u(t))$ with adaptive Dormand-Prince (DOPRI5) and RK4.
3. **Optimal Transport Conditional Flow Matching (OT-CFM)**: Missing cortical sensor channel imputation and uncertainty quantification.
4. **2D Kuramoto Bifurcation Phase Diagram**: Mapping phase synchronization transitions under simulated transcranial magnetic stimulation (TMS).
5. **Edge BCI Benchmark**: Real-time sliding-window inference with sub-millisecond latency.

In [ ]:
# Install dependencies (if running in fresh Google Colab environment)
!pip install -q torch numpy scipy matplotlib pyyaml

In [ ]:
import sys
import os
import math
import time
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F

# Ensure repo path is in python path
sys.path.append(os.path.abspath('..'))
sys.path.append(os.path.abspath('.'))

from src.env.connectome_graph import ConnectomeGraph, WilsonCowanSimulator
from src.models.graph_neural_ode import GraphNeuralODE, odeint_custom
from src.models.flow_matching import OptimalTransportCFM
from src.metrics.evaluators import compute_kuramoto_order_parameter, compute_trajectory_mse, compute_crps

print('PyTorch version:', torch.__version__)
print('All Neural-Flow modules successfully imported.')

## 1. Structural Connectome Graph & Normalized Laplacian

The cortical parcellation consists of $N = 68$ Desikan-Killiany regions organized into four canonical functional subnetworks: Default Mode (DMN), Somatomotor (SMN), Visual (VIS), and Frontoparietal (FPN).

The symmetric normalized Graph Laplacian is defined as:
$$\mathcal{L}_{\mathrm{sym}} = I_N - D^{-1/2} W D^{-1/2}$$
with guaranteed theoretical spectral bounds $\lambda_i(\mathcal{L}_{\mathrm{sym}}) \in [0, 2]$.

In [ ]:
connectome = ConnectomeGraph(num_regions=68, seed=42)
valid, ev = connectome.verify_spectral_bounds()
print(f"Spectral bounds valid [0, 2]: {valid}")
print(f"Smallest eigenvalue: {ev[0]:.6f}, Largest eigenvalue: {ev[-1]:.6f}")

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
im = axes[0].imshow(connectome.adjacency, cmap='magma', vmin=0, vmax=1.0)
axes[0].set_title("Modular Structural Connectivity W")
axes[0].set_xlabel("Region Index")
axes[0].set_ylabel("Region Index")
plt.colorbar(im, ax=axes[0], fraction=0.046)

axes[1].plot(range(len(ev)), ev, 'o-', color='#2b5c8f', markersize=4)
axes[1].axhline(0.0, color='gray', linestyle=':')
axes[1].axhline(2.0, color='red', linestyle='--', label=r'Theoretical Bound $\lambda \leq 2$')
axes[1].set_title("Normalized Laplacian Spectrum")
axes[1].set_xlabel("Eigenmode i")
axes[1].set_ylabel(r"$\lambda_i$")
axes[1].legend()
plt.tight_layout()
plt.show()

## 2. Continuous-Time Graph Neural ODE Simulation

We integrate the non-linear cortical dynamics:
$$\frac{dX(t)}{dt} = -\alpha X(t) - \kappa \mathcal{L}_{\mathrm{sym}} X(t) + f_\theta(X(t), \mathcal{L}_{\mathrm{sym}}, u(t))$$
where $\alpha$ is intrinsic leak damping, $\kappa$ is structural coupling, and $u(t)$ is simulated TMS stimulation on the Frontoparietal network.

In [ ]:
sim = WilsonCowanSimulator(connectome=connectome, coupling_kappa=0.85)
stim = {'t_start': 2.0, 't_end': 4.0, 'amplitude': 0.8, 'frequency': 10.0, 'target_subnetwork': 3}
out = sim.simulate(t_span=(0.0, 6.0), dt=0.02, perturbation=stim, seed=42)

times = out['times']
e_act = out['e_activity']

plt.figure(figsize=(10, 4))
for idx, name, col in [(5, 'DMN R5', '#2b5c8f'), (22, 'SMN R22', '#d95f02'), (58, 'FPN (Stimulated)', '#1b9e77')]:
    plt.plot(times, e_act[:, idx], label=name, color=col, lw=1.8)
plt.axvspan(2.0, 4.0, color='yellow', alpha=0.2, label='Simulated TMS Pulse')
plt.title("Continuous-Time Cortical Population Dynamics")
plt.xlabel("Time (s)")
plt.ylabel("Excitatory Activity E(t)")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## 3. Optimal Transport Conditional Flow Matching (OT-CFM)

OT-CFM learns a conditional probability path from Gaussian noise $X_0 \sim \mathcal{N}(0, I)$ to the cortical distribution $X_1 \sim p_{\mathrm{data}}$ along straight lines:
$$\psi_t(X_0, X_1) = (1 - (1 - \sigma_{\min})t) X_0 + t X_1, \quad u_t(X_1 \mid X_0) = X_1 - (1 - \sigma_{\min}) X_0$$
allowing robust missing sensor channel reconstruction and uncertainty quantification.

In [ ]:
lap_torch = torch.from_numpy(connectome.laplacian).float()
cfm = OptimalTransportCFM(node_dim=4, hidden_dim=32, time_embed_dim=16, num_layers=2)

# Create sample masked cortical observation (mask 35% of channels)
x_true = torch.from_numpy(out['states'][-1:]).float()  # (1, 68, 4)
mask = (torch.rand(1, 68, 1) > 0.35).float()
x_partial = x_true * mask

# Perform multi-sample imputation with uncertainty
mu_pred, std_pred = cfm.impute_missing_channels(x_partial=x_partial, mask=mask, laplacian=lap_torch, steps=10)

print("Imputation completed successfully.")
print(f"Reconstructed tensor shape: {mu_pred.shape}")
print(f"Mean uncertainty std: {std_pred.mean().item():.4f}")

## 4. 2D Kuramoto Bifurcation Phase Diagram

We evaluate the global phase synchronization order parameter:
$$R(t) = \left| \frac{1}{N} \sum_{j=1}^N e^{i \theta_j(t)} \right| \in [0, 1]$$
across structural connectome coupling $\kappa$ and stimulation amplitude $I_{\mathrm{stim}}$.

In [ ]:
phases = out['phase']
r_t, r_mean = compute_kuramoto_order_parameter(phases)
print(f"Mean Kuramoto Order Parameter R: {r_mean:.4f}")

plt.figure(figsize=(9, 3.5))
plt.plot(times, r_t, color='#2b5c8f', lw=2, label=r'Synchronization $R(t)$')
plt.axvspan(2.0, 4.0, color='yellow', alpha=0.2, label='Stimulation Window')
plt.ylim(0, 1.05)
plt.title("Kuramoto Global Phase Synchronization Order Parameter")
plt.xlabel("Time (s)")
plt.ylabel(r"$R(t)$")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## 5. Real-Time BCI Edge Engine Latency Benchmark

We benchmark real-time sliding-window neural state inference for edge Brain-Computer Interface deployment (<5.0 ms target).

In [ ]:
from cpp_inference.run_bci_benchmark import run_benchmark
run_benchmark()